
# Figure 6b transition-SoC map with silicon-OCP deformation coupled to $C_{n,\mathrm{Si}}$

This notebook updates the original Figure 6b map by coupling the silicon-OCP deformation
parameters $s_V$ and $U_{\mathrm{off}}$ to $C_{n,\mathrm{Si}}$ using empirical relations extracted from the
aging dataset.

Workflow:
1. Read the master lifetime-estimation CSV used for the deformation-parameter scatter plots.
2. Build empirical relations from $C_{n,\mathrm{Si}}$ to $s_V$ and $U_{\mathrm{off}}$.
3. Visualize raw data and the fitted/interpolated relations.
4. Recompute the transition-SoC map with and without deformation coupling.
5. Plot the updated map and the difference from the original fixed-$(s_V,U_{\mathrm{off}})$ map.


In [ ]:

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import LinearSegmentedColormap
from matplotlib import rcParams

from scipy.interpolate import PchipInterpolator
from tqdm.auto import tqdm
from joblib import Parallel, delayed

# =========================================================
# project import setup
# =========================================================
def find_repo_root(start=None):
    """Locate the cloned repository so the notebook can run from any folder."""
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "code" / "diagnostic_algorithm_lifetime_crate").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not find the repository root. Run this notebook from inside the cloned repository.")


REPO_ROOT = find_repo_root()
CODE_DIR = REPO_ROOT / "code"
NOTEBOOK_DIR = CODE_DIR / "plotting" / "f6"
DATA_DIR = NOTEBOOK_DIR / "F6_group_plots_outputs_html"

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from diagnostic_algorithm_lifetime_crate.derivative_utils import smooth_then_grad
from diagnostic_algorithm_lifetime_crate.user_functions import (
    Gr_OCP,
    build_effective_si_ocp,
    ocp_3,
)
from diagnostic_algorithm_lifetime_crate.transition_soc_utils import (
    smooth_ocp_curve,
    compute_si_current_share_and_transition_7p,
)

# 1) prepare graphite smooth OCP once
Gr_OCP_smooth = smooth_ocp_curve(
    Gr_OCP,
    x_col="sto",
    y_col="p",
    window_length=51,
    polyorder=3,
    enforce_monotone=True,
)

# =========================================================
# master file and output folder
# =========================================================
master_file = DATA_DIR / "Master_with_groups_and_day.csv"
assert master_file.exists(), f"Master file not found: {master_file}"

# =========================================================
# fixed map settings (same base as the original notebook)
# =========================================================
Qdata_expand = np.linspace(-3, 3, 600)
win = 1
poly = 3
SI_RECONSTRUCTION_MODE = "reconstructed"

# Standard 7-parameter set:
# Internal parameter order:
# [Cn_Si, Cn_Gr, x100, Cp, y100, s_V, U_off]
# where x100 and y100 correspond to x_n,100 and x_p,100 in the manuscript.
base_para = [1.13, 1.465, 0.9614, 2.69, 0.0173, 0.9, 0.034]

# Sweep variables for the map
Cn_Si_list = np.linspace(0.13, 1.13, 10)
x_100_list = np.linspace(0.55, 0.9614, 10)

# Save directory
out_dir = NOTEBOOK_DIR / "figures" / "F6b_coupled_outputs"
out_dir.mkdir(parents=True, exist_ok=True)
print("REPO_ROOT =", REPO_ROOT)
print("out_dir =", out_dir)



In [ ]:

# Load the grouped diagnostic master table and standardize column names used below.
Master = pd.read_csv(master_file)

# aliases consistent with the MATLAB script
if "C" not in Master.columns and "Qcc_max_meas" in Master.columns:
    Master["C"] = Master["Qcc_max_meas"]

if "CnSi" not in Master.columns and "Cn_Si" in Master.columns:
    Master["CnSi"] = Master["Cn_Si"]

if "CnGr" not in Master.columns and "Cn_Gr" in Master.columns:
    Master["CnGr"] = Master["Cn_Gr"]

if "AhTh" not in Master.columns and "Ah_throughput" in Master.columns:
    Master["AhTh"] = Master["Ah_throughput"]

if "EFC" not in Master.columns and "AhTh" in Master.columns:
    Master["EFC"] = Master["AhTh"] / 5

if "cycle_group" in Master.columns:
    Master = Master[Master["cycle_group"].astype(str) != "unassigned"].copy()

required_cols = ["CnSi", "si_scale_a", "si_shift_b"]
missing = [c for c in required_cols if c not in Master.columns]
if missing:
    raise ValueError(f"Missing required columns in master file: {missing}")

df_rel = Master[["CnSi", "si_scale_a", "si_shift_b"]].copy()
df_rel = df_rel.replace([np.inf, -np.inf], np.nan).dropna()

print(df_rel.describe())
print("n valid rows =", len(df_rel))


In [ ]:

# Build empirical silicon-OCP deformation relations using binned medians and PCHIP interpolation.
def build_binned_median_curve(df, x_col, y_col, n_bins=20, min_count=3):
    tmp = df[[x_col, y_col]].dropna().copy()
    x = tmp[x_col].to_numpy()
    y = tmp[y_col].to_numpy()

    order = np.argsort(x)
    x = x[order]
    y = y[order]

    edges = np.linspace(x.min(), x.max(), n_bins + 1)
    mids, meds, counts = [], [], []

    for i in range(n_bins):
        left, right = edges[i], edges[i + 1]
        if i < n_bins - 1:
            mask = (x >= left) & (x < right)
        else:
            mask = (x >= left) & (x <= right)

        if mask.sum() >= min_count:
            mids.append(np.median(x[mask]))
            meds.append(np.median(y[mask]))
            counts.append(mask.sum())

    mids = np.array(mids)
    meds = np.array(meds)
    counts = np.array(counts)

    order2 = np.argsort(mids)
    return mids[order2], meds[order2], counts[order2]

x_sv, y_sv, n_sv = build_binned_median_curve(df_rel, "CnSi", "si_scale_a", n_bins=12, min_count=3)
x_uo, y_uo, n_uo = build_binned_median_curve(df_rel, "CnSi", "si_shift_b", n_bins=12, min_count=3)

sv_interp = PchipInterpolator(x_sv, y_sv, extrapolate=True)
uoff_interp = PchipInterpolator(x_uo, y_uo, extrapolate=True)

CnSi_min = float(df_rel["CnSi"].min())
CnSi_max = float(df_rel["CnSi"].max())

# optional clipping to keep values in a physically reasonable range
SV_MIN, SV_MAX = 0.2, 1.2
UOFF_MIN, UOFF_MAX = 0.0, 0.12

def infer_sv_uoff_from_cnsi(cnsi):
    c = float(np.clip(cnsi, CnSi_min, CnSi_max))
    sv = float(np.clip(sv_interp(c), SV_MIN, SV_MAX))
    uoff = float(np.clip(uoff_interp(c), UOFF_MIN, UOFF_MAX))
    return sv, uoff

x_plot = np.linspace(CnSi_min, CnSi_max, 300)
sv_plot = sv_interp(x_plot)
uoff_plot = uoff_interp(x_plot)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# sV
axes[0].scatter(df_rel["CnSi"], df_rel["si_scale_a"], s=18, alpha=0.5, label="raw")
axes[0].plot(x_sv, y_sv, "o-", label="bin median")
axes[0].plot(x_plot, sv_plot, "-", linewidth=2, label="interp")
axes[0].set_xlabel("CnSi [Ah]")
axes[0].set_ylabel("sV")
axes[0].set_title("CnSi to sV")
axes[0].invert_xaxis()
axes[0].legend()

# U_off
axes[1].scatter(df_rel["CnSi"], df_rel["si_shift_b"], s=18, alpha=0.5, label="raw")
axes[1].plot(x_uo, y_uo, "o-", label="bin median")
axes[1].plot(x_plot, uoff_plot, "-", linewidth=2, label="interp")
axes[1].set_xlabel("CnSi [Ah]")
axes[1].set_ylabel(r"$U_{\mathrm{off}}$ [V]")
axes[1].set_title(r"$C_{n,\mathrm{Si}}$ to $U_{\mathrm{off}}$")
axes[1].invert_xaxis()
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:

# Evaluate transition SoC maps with fixed deformation and with deformation coupled to CnSi.
def eval_one_point_fixed(i, j, x_100, Cn_Si):
    params = [
        Cn_Si,
        base_para[1],
        x_100,
        base_para[3],
        base_para[4],
        base_para[5],
        base_para[6],
    ]

    try:
        res = compute_si_current_share_and_transition_7p(
            params=params,
            Qdata_expand=Qdata_expand,
            ocp_3=ocp_3,
            smooth_then_grad=smooth_then_grad,
            build_effective_si_ocp=build_effective_si_ocp,
            Gr_OCP_smooth=Gr_OCP_smooth,
            SI_RECONSTRUCTION_MODE=SI_RECONSTRUCTION_MODE,
            win=win,
            poly=poly,
            threshold=0.5,
            persistence_window=0.10,
            tol=0.01,
        )
        transition_soc = res["transition_soc"]
        transition_reason = res["transition_reason"]
        value = transition_soc * 100.0 if np.isfinite(transition_soc) else np.nan
        return i, j, value, transition_reason, base_para[5], base_para[6]
    except Exception as e:
        return i, j, np.nan, f"ERROR: {repr(e)}", np.nan, np.nan


def eval_one_point_coupled(i, j, x_100, Cn_Si):
    sV_here, Uoff_here = infer_sv_uoff_from_cnsi(Cn_Si)
    params = [
        Cn_Si,
        base_para[1],
        x_100,
        base_para[3],
        base_para[4],
        sV_here,
        Uoff_here,
    ]

    try:
        res = compute_si_current_share_and_transition_7p(
            params=params,
            Qdata_expand=Qdata_expand,
            ocp_3=ocp_3,
            smooth_then_grad=smooth_then_grad,
            build_effective_si_ocp=build_effective_si_ocp,
            Gr_OCP_smooth=Gr_OCP_smooth,
            SI_RECONSTRUCTION_MODE=SI_RECONSTRUCTION_MODE,
            win=win,
            poly=poly,
            threshold=0.5,
            persistence_window=0.10,
            tol=0.01,
        )
        transition_soc = res["transition_soc"]
        transition_reason = res["transition_reason"]
        value = transition_soc * 100.0 if np.isfinite(transition_soc) else np.nan
        return i, j, value, transition_reason, sV_here, Uoff_here
    except Exception as e:
        return i, j, np.nan, f"ERROR: {repr(e)}", sV_here, Uoff_here


def build_map(eval_fn, tag):
    soc_matrix = np.full((len(x_100_list), len(Cn_Si_list)), np.nan)
    reason_matrix = np.empty((len(x_100_list), len(Cn_Si_list)), dtype=object)
    sv_map = np.full((len(x_100_list), len(Cn_Si_list)), np.nan)
    uoff_map = np.full((len(x_100_list), len(Cn_Si_list)), np.nan)

    tasks = [(i, j, x_100, Cn_Si)
             for i, x_100 in enumerate(x_100_list)
             for j, Cn_Si in enumerate(Cn_Si_list)]

    results = Parallel(n_jobs=-1, backend="loky")(
        delayed(eval_fn)(i, j, x_100, Cn_Si)
        for (i, j, x_100, Cn_Si) in tqdm(tasks, desc=f"Computing {tag} map")
    )

    for i, j, value, reason, sv_here, uoff_here in results:
        soc_matrix[i, j] = value
        reason_matrix[i, j] = reason
        sv_map[i, j] = sv_here
        uoff_map[i, j] = uoff_here

    return soc_matrix, reason_matrix, sv_map, uoff_map


In [ ]:

# Compute the fixed-deformation and coupled-deformation maps.
SoC_matrix_fixed, reason_fixed, sV_map_fixed, Uoff_map_fixed = build_map(eval_one_point_fixed, "fixed_ab")
SoC_matrix_coupled, reason_coupled, sV_map_coupled, Uoff_map_coupled = build_map(eval_one_point_coupled, "coupled_ab")

pd.DataFrame(SoC_matrix_fixed, index=np.round(x_100_list, 6), columns=np.round(Cn_Si_list, 6)).to_csv(out_dir / "transition_soc_map_fixed_ab.csv")
pd.DataFrame(SoC_matrix_coupled, index=np.round(x_100_list, 6), columns=np.round(Cn_Si_list, 6)).to_csv(out_dir / "transition_soc_map_coupled_ab.csv")
pd.DataFrame(sV_map_coupled, index=np.round(x_100_list, 6), columns=np.round(Cn_Si_list, 6)).to_csv(out_dir / "sV_map_coupled.csv")
pd.DataFrame(Uoff_map_coupled, index=np.round(x_100_list, 6), columns=np.round(Cn_Si_list, 6)).to_csv(out_dir / "Uoff_map_coupled.csv")

print("Saved raw map CSVs to", out_dir)


In [ ]:

# Inspect the coupled deformation fields over the map grid.
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

im0 = axes[0].imshow(sV_map_coupled, origin="lower", aspect="auto")
axes[0].set_title(r"Coupled $s_V$ over map grid")
axes[0].set_xlabel("CnSi index")
axes[0].set_ylabel(r"$x_{n,100}$ index")
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(Uoff_map_coupled, origin="lower", aspect="auto")
axes[1].set_title(r"Coupled $U_{\mathrm{off}}$ over map grid")
axes[1].set_xlabel("CnSi index")
axes[1].set_ylabel(r"$x_{n,100}$ index")
plt.colorbar(im1, ax=axes[1])

plt.tight_layout()
plt.savefig(out_dir / "coupled_sv_uoff_maps.png", dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", out_dir / "coupled_sv_uoff_maps.png")


In [ ]:

# Convert the sweep axes into the LAM-Si and LLI coordinates used in the figure.
Cn_Si_base, Cn_Gr_base, x100_base, Cp_base, y100_base, sV_base, Uoff_base = base_para
LAM_Si = Cn_Si_base - Cn_Si_list

Li_max = x100_base * (Cn_Si_base + Cn_Gr_base) + y100_base * Cp_base
LLI = np.zeros((len(x_100_list), len(Cn_Si_list)))
for i, x_100 in enumerate(x_100_list):
    for j, Cn_Si in enumerate(Cn_Si_list):
        LLI[i, j] = Li_max - (x_100 * (Cn_Si + Cn_Gr_base) + y100_base * Cp_base)

X = LLI
Y = np.tile(LAM_Si.reshape(1, -1), (len(x_100_list), 1))
Z_fixed = np.ma.masked_invalid(SoC_matrix_fixed)
Z_coupled = np.ma.masked_invalid(SoC_matrix_coupled)
Z_delta = np.ma.masked_invalid(SoC_matrix_coupled - SoC_matrix_fixed)


In [ ]:

# Plotting utilities for transition-SoC maps.
dark_hex  = "#55748f"
light_hex = "#c36439"
custom_cmap = LinearSegmentedColormap.from_list(
    "blue_to_bronze",
    [mcolors.to_rgb(dark_hex), mcolors.to_rgb(light_hex)],
    N=256
)
custom_cmap.set_bad(color="white")

rcParams.update({
    "text.usetex": False,
    "font.family": "sans-serif",
    "font.sans-serif": ["Helvetica", "Arial", "DejaVu Sans"],
    "svg.fonttype": "none",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.grid": False,
    "axes.linewidth": 0.5,
    "xtick.major.width": 0.5,
    "ytick.major.width": 0.6,
    "xtick.major.size": 3,
    "ytick.major.size": 3,
})

def plot_transition_map(X, Y, Z, title=None, save_name=None):
    fig, ax = plt.subplots(figsize=(4.4, 2.8))
    cf = ax.contourf(X, Y, Z, levels=40, cmap=custom_cmap)

    ax.set_xlim(0.0, 1)
    ax.set_ylim(0.0, 1)

    levels_lines = [10, 20, 30, 40, 50, 60, 70, 80]
    cs = ax.contour(X, Y, Z, levels=levels_lines, colors="white", linewidths=0.5)

    manual_positions = [
        (0.82, 0.78),  # 10
        (0.74, 0.68),  # 20
        (0.55, 0.47),  # 30
        (0.95, 0.66),  # 40
        # (0.58, 0.40),  # 50
        (0.86, 0.53),  # 60
        (0.84, 0.12),  # 70
        # (0.90, 0.02),  # 80
    ]
    ax.clabel(cs, inline=True, fmt="%d", fontsize=12, colors="white",
              manual=manual_positions, inline_spacing=2)

    cbar = fig.colorbar(cf, ax=ax, fraction=0.045, pad=0.03)
    cbar.set_label("Transition SoC [%]", fontsize=14)
    cbar.ax.tick_params(labelsize=14, direction="in", width=0.5, length=3)
    for spine in cbar.ax.spines.values():
        spine.set_linewidth(0.5)

    ax.set_xlabel("LLI [Ah]", fontsize=14)
    ax.set_ylabel("LAM Si [Ah]", fontsize=14)
    if title is not None:
        ax.set_title(title, fontsize=15)
    ax.tick_params(axis="both", direction="in", labelsize=14, width=0.5, length=3)
    for spine in ax.spines.values():
        spine.set_linewidth(0.5)

    plt.tight_layout(pad=0.4)
    if save_name is not None:
        plt.savefig(out_dir / save_name, dpi=300, bbox_inches="tight")
        plt.savefig(out_dir / save_name.replace('.png', '.svg'), bbox_inches="tight")
    plt.show()


In [ ]:

# Generate the final transition-SoC map figures.
plot_transition_map(X, Y, Z_fixed, save_name="transition_soc_map_fixed_ab.png")
plot_transition_map(X, Y, Z_coupled, save_name="transition_soc_map_coupled_ab.png")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

im0 = axes[0].imshow(SoC_matrix_fixed, origin="lower", aspect="auto")
axes[0].set_title("Old map")
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(SoC_matrix_coupled, origin="lower", aspect="auto")
axes[1].set_title("Coupled map")
plt.colorbar(im1, ax=axes[1])

im2 = axes[2].imshow(SoC_matrix_coupled - SoC_matrix_fixed, origin="lower", aspect="auto")
axes[2].set_title("Difference (new - old)")
plt.colorbar(im2, ax=axes[2])

for ax in axes:
    ax.set_xlabel("CnSi index")
    ax.set_ylabel(r"$x_{n,100}$ index")

plt.tight_layout()
plt.savefig(out_dir / "transition_soc_map_old_vs_new_vs_delta.png", dpi=300, bbox_inches="tight")
plt.show()

print("Saved map figures to", out_dir)


In [ ]:
nan_mask_fixed = np.isnan(SoC_matrix_fixed)
nan_mask_coupled = np.isnan(SoC_matrix_coupled)

print("Fixed map NaN reasons:")
print(pd.Series(reason_fixed[nan_mask_fixed].ravel()).value_counts().head(20))

print("\nCoupled map NaN reasons:")
print(pd.Series(reason_coupled[nan_mask_coupled].ravel()).value_counts().head(20))